# EasyMagpieTTS — vLLM-Omni inference demo (dummy weights)

This notebook runs an end-to-end inference pass through the
[`easymagpie_vllm_omni`](./easymagpie_vllm_omni) model definition using
**dummy / random weights**, so you can exercise the full engine path
(prefill -> autoregressive decode -> audio-code extraction) without a converted
checkpoint.

It follows the same `AsyncOmni` single-stage pattern as the reference
`qwen3-tts` and `eartts` demos:

* **prefill** — the caller supplies the speaker-encoded context-audio embedding
  via `additional_information.speaker_embedding` `(T_audio, embedding_dim)` plus a
  plain `context_text` string; the model assembles the full prefill context
  (`[task_embedding? | speaker_embedding | context_text_embedded]`) and tokenizes
  `context_text` itself. `prompt_token_ids = [0] * prompt_len`, sized with
  `EasyMagpieTTSForConditionalGeneration.estimate_prompt_len(...)`.
* **decode** — each step consumes one subword id from the streaming
  `additional_information.text_tokens` list; the local transformer samples all
  `C * S` stacked audio codebooks for the frame.
* **output** — per-step audio codes are surfaced on
  `OmniOutput.multimodal_outputs[\"audio_codes\"]` (`BT x num_codebooks`), and the
  engine accumulates them across steps just like eartts, so we trim to the last
  `len(token_ids)` decoded rows.

> **Dummy weights.** We build a `config.json` sized to the real checkpoint
> (`2605_EMTTS_SmallMamba_Step150k_posttrained_epoch12.nemo`) and start the
> engine with `load_format=\"dummy\"`, so vLLM fills all parameters with random
> values. The emitted codes are therefore meaningless — this is a *smoke test*
> of the engine wiring, not a real synthesis. Point the engine at a real
> converted checkpoint (and drop `load_format`) to get audio.

> **Environment.** Run this inside the bootstrapped `vllm_omni_env` (vLLM +
> vLLM-Omni + compatible torch) with the plugin installed:
> ```bash
> source /path/to/vllm_omni_env/bin/activate
> pip install -e examples/tts/easymagpie_vllm_omni
> ```

In [ ]:
import os

# Single-process executor below, but keep spawn semantics consistent with the
# qwen3-tts / eartts demos in case you switch to a multiproc backend.
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

import json
import tempfile
import uuid
from pathlib import Path

import torch
import yaml

from vllm import SamplingParams
from vllm_omni import AsyncOmni

# Importing the model package is optional (the engine resolves the arch via the
# `vllm.general_plugins` entry point installed with the package), but doing it
# here surfaces the arch dataclass we use to size the dummy prompt embedding.
from easymagpie_vllm_omni.config import EasyMagpieOmniArch

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 1. Build a tiny dummy model directory

The engine only needs a `config.json` that (a) names the registered arch and
(b) carries the EasyMagpie + Nemotron-H scalars. Here we size everything to match
the real checkpoint
`2605_NemotronTTS_V0.2/v2/2605_EMTTS_SmallMamba_Step150k_posttrained_epoch12.nemo`
(hidden 1536, 8 codebooks × 1024, frame-stacking ×2, 3-layer local transformer).

The backbone is a **Nemotron-H** hybrid (Mamba2 + attention + MoE) decoder:
`EasyMagpieTTSForConditionalGeneration` constructs vLLM's `NemotronHModel` and
implements the hybrid-Mamba interfaces (`HasInnerState` / `IsHybrid` /
`SupportsMambaPrefixCaching`), exactly like the EasyMagpie vLLM *sidecar*. The
`nemotron_h_config` fields (`hybrid_override_pattern`, `mamba_*`, `n_routed_experts`,
…) are copied verbatim from the checkpoint.

The EasyMagpie-specific scalars (`embedding_dim`, `num_audio_codebooks`,
`codebook_size`, `frame_stacking_factor`, `local_transformer_*`, …) are read by
`EasyMagpieOmniArch.from_hf_config`. The phoneme branch is **enabled**
(`phoneme_stacking_factor = 1`, `phoneme_vocab_size = 2051`) to match the
checkpoint; the model self-predicts phonemes, so no phoneme stream needs to be
supplied in the prompt.

With `load_format=\"dummy\"` (set in the stage config) vLLM never reads weight
files, so no safetensors are needed. We do save the checkpoint's
text-conditioning tokenizer (`TEXT_TOKENIZER`, the Nemotron-H tokenizer that
matches `TEXT_VOCAB`) into the model dir, since the model tokenizes the
per-request `context_text` in-engine via
`AutoTokenizer.from_pretrained(model_path)`.

In [ ]:
# Config matching the real checkpoint:
#   2605_NemotronTTS_V0.2/v2/2605_EMTTS_SmallMamba_Step150k_posttrained_epoch12.nemo
#
# The backbone is a Nemotron-H hybrid (Mamba2 + attention + MoE) decoder, wired
# through vLLM's `NemotronHModel` by `EasyMagpieTTSForConditionalGeneration`. The
# fields below are ported verbatim from the checkpoint's `model_config.yaml`
# (the `nemotron_h_config` block + the EasyMagpie scalars). With
# `load_format="dummy"` the weights are random — a realistically-sized smoke test.
#
#   embedding_dim == hidden_size == audio_embedding_dim == local_transformer_hidden_dim
#   (all 1536 in the checkpoint) keeps every in/out projection an Identity.
HIDDEN = 1536            # nemotron_h_config.hidden_size / embedding_dim / audio_embedding_dim
NUM_AUDIO_CODEBOOKS = 8  # vector_quantizer.num_groups
CODEBOOK_SIZE = 1024     # prod(vector_quantizer.num_levels_per_group) = 4**5
FRAME_STACKING = 2       # -> num_stacked_codebooks = NUM_AUDIO_CODEBOOKS * FRAME_STACKING = 16
PHONEME_STACKING = 1     # phoneme_stacking_factor
PHONEME_VOCAB = 2051     # IPA-BPE 2048 tokenizer + 3 special tokens
TEXT_VOCAB = 131072      # nemotron_h_config.vocab_size
# Text-conditioning tokenizer that matches the checkpoint (SmallMamba uses the
# Nemotron-H tokenizer, vocab 131072 == TEXT_VOCAB). Point this at the converted
# checkpoint dir / the checkpoint's tokenizer when running a real model.
TEXT_TOKENIZER = "nvidia/Nemotron-H-8B-Base-8K"

config = {
    # Resolved through the `vllm.general_plugins` entry point registered by the
    # `easymagpie_vllm_omni` package -> EasyMagpieTTSForConditionalGeneration.
    "architectures": ["EasyMagpieTTSForConditionalGeneration"],
    # Nemotron-H backbone fields (consumed by vllm NemotronHModel) — copied
    # verbatim from the checkpoint's `nemotron_h_config` block.
    "model_type": "nemotron_h",
    "hidden_size": HIDDEN,
    "num_hidden_layers": 31,
    "vocab_size": TEXT_VOCAB,
    "num_attention_heads": 12,
    "num_key_value_heads": 4,
    "attention_dropout": 0.0,
    "attention_bias": False,
    "max_position_embeddings": 8192,
    "mamba_num_heads": 64,
    "mamba_head_dim": 24,
    "ssm_state_size": 128,
    "conv_kernel": 4,
    "n_groups": 8,
    "chunk_size": 256,
    "mamba_hidden_act": "silu",
    "use_conv_bias": True,
    "use_bias": False,
    "intermediate_size": 4096,
    "mlp_hidden_act": "silu",
    "mlp_bias": False,
    "n_routed_experts": 24,
    "num_experts_per_tok": 4,
    "moe_intermediate_size": 768,
    "moe_shared_expert_intermediate_size": 2048,
    "n_group": 1,
    "topk_group": 1,
    "routed_scaling_factor": 2.5,
    "norm_topk_prob": True,
    # 31-char layer pattern: M=Mamba2, *=attention, E=MLP/MoE (len == num_hidden_layers).
    "hybrid_override_pattern": "MEMEM*EMEMEM*EMEMEMEM*EMEMEMEME",
    "layer_norm_epsilon": 1e-5,
    "residual_in_fp32": False,
    "tie_word_embeddings": False,
    # bfloat16, not float32: the Nemotron-H MoE layers run vLLM's fused-MoE
    # Triton kernel, whose block sizes are tuned for 16-bit. In float32 the
    # kernel needs ~2x shared memory and overflows the GPU limit
    # (OutOfResources: shared memory). bf16 also matches the real checkpoint.
    "torch_dtype": "bfloat16",
    # EasyMagpie-specific scalars (read by EasyMagpieOmniArch.from_hf_config).
    "text_vocab_size": TEXT_VOCAB,
    "embedding_dim": HIDDEN,
    "audio_embedding_dim": HIDDEN,
    "num_audio_codebooks": NUM_AUDIO_CODEBOOKS,
    "codebook_size": CODEBOOK_SIZE,
    "frame_stacking_factor": FRAME_STACKING,
    "phoneme_stacking_factor": PHONEME_STACKING,
    "phoneme_vocab_size": PHONEME_VOCAB,
    "local_transformer_n_layers": 3,
    "local_transformer_n_heads": 12,
    "local_transformer_hidden_dim": HIDDEN,
}

MODEL_DIR = Path(tempfile.mkdtemp(prefix="easymagpie_dummy_"))
(MODEL_DIR / "config.json").write_text(json.dumps(config, indent=2))

# The model tokenizes the per-request `context_text` string in-engine via
# `AutoTokenizer.from_pretrained(model_path)` (qwen3-tts style), so the model dir
# must ship the checkpoint's text-conditioning tokenizer. We save the matching
# Nemotron-H tokenizer (TEXT_TOKENIZER) into MODEL_DIR.
from transformers import AutoTokenizer

AutoTokenizer.from_pretrained(TEXT_TOKENIZER, trust_remote_code=True).save_pretrained(MODEL_DIR)
print(f"Dummy model dir: {MODEL_DIR}")

# Sanity-check the arch the model will derive from this config.
arch = EasyMagpieOmniArch.from_hf_config(type("Cfg", (), config))
print(f"embedding_dim            : {arch.embedding_dim}")
print(f"num_stacked_codebooks    : {arch.num_stacked_codebooks}  (C*S)")
print(f"tokens / codebook        : {arch.num_all_tokens_per_codebook}  (codebook_size + specials)")
print(f"audio_bos / audio_eos id : {arch.audio_bos_id} / {arch.audio_eos_id}")

## 2. Single-stage `AsyncOmni` engine

A single `llm` stage that runs the EasyMagpie talker, mirroring the eartts demo
(`worker_type=\"ar\"`, `OmniARScheduler`). The stage declares
`engine_output_type=\"audio\"` / `final_output_type=\"audio\"`: for a single-stage
AR TTS model these make the runner attach the per-step `audio_codes` multimodal
payload to the output (with `\"latent\"` the payload is dropped because nothing
downstream consumes it, and `multimodal_output[\"audio_codes\"]` comes back
`None`). Two extra knobs make this a dummy-weights run with no external assets:

* `load_format: \"dummy\"` — vLLM initializes random weights instead of reading a
  checkpoint (so `load_weights` / `init_forbidden_mask` are skipped; the
  forbidden-token mask stays all-zeros, i.e. no sampling mask — fine for a smoke
  test).
* `skip_tokenizer_init: true` — we feed `prompt_token_ids` + `text_tokens`
  directly, so no tokenizer files are needed.

`max_model_len` must cover `T_ctx` (prefill) + the number of decode steps.

In [ ]:
DECODE_STEPS = 32     # number of audio frames to decode
# Prefill length is derived at prompt-build time from the speaker embedding +
# tokenized context_text (see the prompt cell); these just need to be large
# enough to cover prefill + decode.
MAX_MODEL_LEN = 512
MAX_NUM_BATCHED_TOKENS = 512

stage_cfg = {
    "stage_args": [
        {
            "stage_id": 0,
            "stage_type": "llm",
            "is_comprehension": True,
            "final_output": True,
            # "audio" (not "latent") is required for a single-stage AR TTS model:
            # it makes the AR model runner attach the per-step multimodal payload
            # ("audio_codes") to the EngineCoreOutput even though no downstream
            # stage consumes it, so the codes reach the client. With "latent" the
            # payload is dropped and multimodal_output["audio_codes"] is None.
            "final_output_type": "audio",
            "runtime": {"devices": "0"},
            "engine_args": {
                "model_stage": "easymagpie",
                "max_num_seqs": 1,
                "model_arch": "EasyMagpieTTSForConditionalGeneration",
                "worker_type": "ar",
                "scheduler_cls": "vllm_omni.core.sched.omni_ar_scheduler.OmniARScheduler",
                #"enforce_eager": True,  # dummy run: skip CUDA-graph capture for a faster start
                "trust_remote_code": True,
                "async_scheduling": True,
                "enable_prefix_caching": False,
                "engine_output_type": "audio",
                "gpu_memory_utilization": 0.6,
                "distributed_executor_backend": "uni",
                "max_num_batched_tokens": MAX_NUM_BATCHED_TOKENS,
                "max_model_len": MAX_MODEL_LEN,
                # bf16 (not fp32): the Nemotron-H fused-MoE Triton kernel's block
                # sizes are tuned for 16-bit and overflow shared memory in fp32.
                "dtype": "bfloat16",
                "attention_backend": "TRITON_ATTN",
                # --- dummy-weights smoke-test knobs ---
                "load_format": "dummy",
                "skip_tokenizer_init": True,
            },
            "default_sampling_params": {
                "temperature": 0.0,
                "max_tokens": DECODE_STEPS,
                "detokenize": False,
                "ignore_eos": True,
            },
        }
    ],
}

_tmp = tempfile.NamedTemporaryFile(
    mode="w", suffix=".yaml", prefix="easymagpie_omni_demo_", delete=False,
)
yaml.dump(stage_cfg, _tmp, sort_keys=False)
_tmp.close()
STAGE_CFG_PATH = _tmp.name
print(f"Stage config: {STAGE_CFG_PATH}")

omni = AsyncOmni(
    model=str(MODEL_DIR),
    stage_configs_path=STAGE_CFG_PATH,
    log_stats=False,
    stage_init_timeout=300,
)
print("Engine ready (single stage: EasyMagpie talker, dummy weights)")

## 3. Build the prompt

Per-request input, passed through `additional_information`:

* **`speaker_embedding`** `(T_audio, embedding_dim)` — the speaker-encoded
  context-audio embedding (the audio branch of `prepare_context_tensors`),
  loaded here from `eng_speaker_emb.pt` (as written by
  `easy_magpietts_extract_speaker_encoding.py`). The model assembles the full
  prefill context itself as `[task_embedding? | speaker_embedding |
  context_text_embedded]`.
* **`context_text`** — a plain conditioning string, here `"[EN]"`. The model
  tokenizes it in-engine and embeds it through the baked `text_embedding` table.
* **`text_tokens`** `list[int]` — the streaming subword stream; decode step `k`
  consumes `text_tokens[k]`. We provide one id per decode step.

`prompt_token_ids = [0] * prompt_len` are placeholders (the model feeds the
backbone via `inputs_embeds`, never these ids). `prompt_len` must equal the
assembled context length, so we size it with the model's
`estimate_prompt_len(...)` — the length-only mirror of the in-engine prefill
assembly (à la qwen3-tts's `estimate_prompt_len_from_additional_information`).

(If the checkpoint had a phoneme branch you'd also stream `phoneme_tokens`.)

In [ ]:
torch.manual_seed(0)

from transformers import AutoTokenizer

from easymagpie_vllm_omni.easymagpie import EasyMagpieTTSForConditionalGeneration

# Speaker-encoded context audio (audio branch of prepare_context_tensors),
# produced by easy_magpietts_extract_speaker_encoding.py.
SPEAKER_EMB_FILE = "eng_speaker_emb.pt"
_loaded = torch.load(SPEAKER_EMB_FILE, map_location="cpu")
speaker_embedding = _loaded["speaker_encoding"] if isinstance(_loaded, dict) else _loaded
speaker_embedding = speaker_embedding.to(torch.float32)

# Plain conditioning string; the model tokenizes + embeds it in-engine.
CONTEXT_TEXT = "[EN]"

# Same tokenizer the engine loads from MODEL_DIR — used to size the prefill
# placeholders so prompt_token_ids length matches the assembled context.
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
prompt_len = EasyMagpieTTSForConditionalGeneration.estimate_prompt_len(
    speaker_embedding,
    tokenize=lambda t: tokenizer.encode(t),
    context_text=CONTEXT_TEXT,
    has_task_embedding=arch.num_task_embeddings > 0,
)

# Streaming subword ids: one per decode step (step k consumes text_tokens[k]).
text_tokens = torch.randint(0, TEXT_VOCAB, (DECODE_STEPS,)).tolist()

additional_information = {
    "speaker_embedding": speaker_embedding,  # (T_audio, embedding_dim) tensor
    "context_text": CONTEXT_TEXT,            # plain string, tokenized in-model
    "text_tokens": text_tokens,              # list[int], grows by one per step
}

prompt = {
    "prompt_token_ids": [0] * prompt_len,    # prefill placeholders
    "additional_information": additional_information,
}

assert prompt_len + DECODE_STEPS <= MAX_MODEL_LEN, (
    f"prompt_len ({prompt_len}) + decode steps ({DECODE_STEPS}) exceeds "
    f"MAX_MODEL_LEN ({MAX_MODEL_LEN}); raise MAX_MODEL_LEN / MAX_NUM_BATCHED_TOKENS."
)

print(f"speaker_embedding            : {tuple(speaker_embedding.shape)}")
print(f"context_text                 : {CONTEXT_TEXT!r} -> {tokenizer.encode(CONTEXT_TEXT)}")
print(f"prompt_len (placeholders)    : {prompt_len}")
print(f"decode steps (max_tokens)    : {DECODE_STEPS}")
print(f"text_tokens[:8]              : {text_tokens[:8]}")

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=DECODE_STEPS,
    detokenize=False,
    ignore_eos=True,  # dummy logits never emit a meaningful EOS -> run the full budget
)

## 4. Run inference and extract audio codes

`omni.generate(...)` is an async generator yielding one `RequestOutput` per
engine step; we keep the last one. As in the eartts demo, the accumulated
`multimodal_output[\"audio_codes\"]` holds one row per flat-batch token over the
whole run (the `T_ctx` prefill frames — codes left zero — plus one frame per
decode step), so we trim to the last `len(token_ids)` rows to recover just the
decoded frames.

In [ ]:
async def run_request(prompt: dict, sampling_params):
    request_id = f"easymagpie-{uuid.uuid4().hex[:8]}"
    final_ro = None
    num_steps = 0
    async for stage_output in omni.generate(
        prompt,
        sampling_params_list=[sampling_params],
        request_id=request_id,
    ):
        final_ro = stage_output
        num_steps += 1
    return final_ro, num_steps


final_ro, num_steps = await run_request(prompt, sampling_params)
assert final_ro is not None, "no output from engine"

mm = final_ro.multimodal_output or {}
audio_codes = mm.get("audio_codes")
token_ids = final_ro.outputs[0].token_ids if final_ro.outputs else []

print(f"Engine steps yielded       : {num_steps}")
print(f"Layer-0 tokens (token_ids) : {len(token_ids)}")
if isinstance(audio_codes, torch.Tensor):
    audio_codes = audio_codes.detach().cpu().to(torch.long)
    print(f"audio_codes shape (raw)    : {tuple(audio_codes.shape)}")
    # Trim the Tref prefill frames echoed during prefill: keep only the decoded
    # frames (the last len(token_ids) rows), exactly like the eartts demo.
    if len(token_ids) > 0:
        audio_codes = audio_codes[-len(token_ids):].contiguous()
    print(f"audio_codes shape (decode) : {tuple(audio_codes.shape)}")
    print(f"audio_codes dtype          : {audio_codes.dtype}")
    print(f"codes min / max            : {int(audio_codes.min())} / {int(audio_codes.max())}")
    print(f"first frame codes          : {audio_codes[0].tolist()}")
else:
    print(f"audio_codes                : {audio_codes!r}")

In [ ]:
import matplotlib.pylab as plt

plt.imshow(audio_codes.T, aspect="auto")
plt.colorbar()
plt.show()